# 📊 Factor Attribution Engine — Multi-Regional + Dashboard
### US · Europe · Emerging Markets · Global Multi-Asset

---

## Multi-Regional Architecture

```
┌─────────────────────────────────────────────────────────────┐
│  REGION DETECTION  →  automatic factor selection            │
│                                                             │
│  US-heavy   → FF5-US    + FRED-USD macro + USD ETFs         │
│  EU-heavy   → FF5-EU    + ECB/FRED macro + Xetra/EUR ETFs   │
│  EM-heavy   → FF5-EM    + EM-specific macro + EM ETFs       │
│  Global     → FF5-Global + all regional factors             │
└─────────────────────────────────────────────────────────────┘
         ↓
  ETF Orthogonalisation → OLS HC3 Regression
         ↓
  Interactive Dash Dashboard (6 sections)
```

---
**Data sources:**
- 📈 **Yahoo Finance** — ETF/equity prices
- 🏛️ **Ken French Library** — FF5 for US, Europe, Developed, Emerging
- 🏦 **FRED** — USD macro (Fed, CPI, spreads)
- 🇪🇺 **ECB SDW** — EUR macro (EUR curve, HICP, BTP-Bund spread)
- 🌍 **ETF proxies** — 70+ multi-regional orthogonalised ETFs


In [83]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import yfinance as yf
import requests, io, zipfile, warnings, os
from datetime import datetime
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas_datareader.data as web

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)


In [84]:
# ══════════════════════════════════════════════════════════════
# ✏️  PORTFOLIO — edit here or load from CSV
# ══════════════════════════════════════════════════════════════
PORTFOLIO = {
    # US Equity
    'SPY':   {'weight': 1, 'name': 'S&P 500',              'asset_class': 'equity',    'region': 'US'},
    'QQQ':   {'weight': 0.0, 'name': 'Nasdaq 100',           'asset_class': 'equity',    'region': 'US'},
    # European Equity
    'EZU':   {'weight': 0.00, 'name': 'MSCI Eurozone',        'asset_class': 'equity',    'region': 'EU'},
    'EWG':   {'weight': 0.00, 'name': 'MSCI Germany',         'asset_class': 'equity',    'region': 'EU'},
    # EM Equity
    'EEM':   {'weight': 0.00, 'name': 'MSCI EM',              'asset_class': 'equity',    'region': 'EM'},
    'FXI':   {'weight': 0.00, 'name': 'China Large Cap',      'asset_class': 'equity',    'region': 'EM'},
    # Bonds
    'TLT':   {'weight': 0.00, 'name': '20Y Treasury',         'asset_class': 'bond',      'region': 'US'},
    'IEF':   {'weight': 0.00, 'name': '7-10Y Treasury',       'asset_class': 'bond',      'region': 'US'},
    'EMB':   {'weight': 0.00, 'name': 'EM Sovereign Bond',    'asset_class': 'bond',      'region': 'EM'},
    'IBGL.L':{'weight': 0.00, 'name': 'EUR Govt Bond',        'asset_class': 'bond',      'region': 'EU'},
    # Alternatives
    'GLD':   {'weight': 0.00, 'name': 'Gold',                 'asset_class': 'commodity', 'region': 'Global'},
    'VNQ':   {'weight': 0.00, 'name': 'US REITs',             'asset_class': 'real_estate','region': 'US'},
    'LQD':   {'weight': 0.00, 'name': 'IG Corp Bond',         'asset_class': 'bond',      'region': 'US'},
}

# Load from CSV (uncomment)
# df_csv = pd.read_csv('portfolio.csv')
# PORTFOLIO = {r['ticker']: {k: r[k] for k in ['weight','name','asset_class','region']} for _,r in df_csv.iterrows()}


In [85]:
# ══════════════════════════════════════════════════════════════
# ⚙️  PARAMETERS
# ══════════════════════════════════════════════════════════════
START_DATE      = '2020-01-01'
END_DATE        = datetime.today().strftime('%Y-%m-%d')
BENCHMARK       = '^GSPC'    # fallback benchmark
FREQUENCY       = 'W'
MAX_ETF_FACTORS = 25
DASHBOARD_PORT  = 8050

# Normalise weights
tw = sum(v['weight'] for v in PORTFOLIO.values())
for k in PORTFOLIO: PORTFOLIO[k]['weight'] /= tw

In [86]:
# ══════════════════════════════════════════════════════════════
# 🌍 REGION DETECTION from portfolio
# ══════════════════════════════════════════════════════════════
region_weights = {}
for t, v in PORTFOLIO.items():
    r = v.get('region', 'US')
    region_weights[r] = region_weights.get(r, 0) + v['weight']

dominant_region = max(region_weights, key=region_weights.get)
is_global = len([r for r in region_weights if region_weights[r] > 0.15]) >= 3
model_type = 'Global' if is_global else dominant_region

print('✅ Portfolio configured:')
df_port_view = pd.DataFrame(PORTFOLIO).T
df_port_view['weight'] = df_port_view['weight'].map('{:.1%}'.format)
display(df_port_view)

print(f'\n🌍 Regional composition:')
for r, w in sorted(region_weights.items(), key=lambda x: -x[1]):
    bar = '█' * int(w * 50)
    print(f'   {r:<8} {w:.1%}  {bar}')
print(f'\n   → Selected model: [{model_type}]')

✅ Portfolio configured:


,weight,name,asset_class,region
SPY,100.0%,S&P 500,equity,US
QQQ,0.0%,Nasdaq 100,equity,US
EZU,0.0%,MSCI Eurozone,equity,EU
EWG,0.0%,MSCI Germany,equity,EU
EEM,0.0%,MSCI EM,equity,EM
FXI,0.0%,China Large Cap,equity,EM
TLT,0.0%,20Y Treasury,bond,US
IEF,0.0%,7-10Y Treasury,bond,US
EMB,0.0%,EM Sovereign Bond,bond,EM
IBGL.L,0.0%,EUR Govt Bond,bond,EU



🌍 Regional composition:
   US       100.0%  ██████████████████████████████████████████████████
   EU       0.0%  
   EM       0.0%  
   Global   0.0%  

   → Selected model: [US]


In [87]:
# ============================================================
# CELL 3 — Multi-Regional ETF Universe (70+ tickers)
# ============================================================

ETF_UNIVERSE = {
    # ── US SECTOR ────────────────────────────────────────────
    'us_sector': {
        'XLK':'Tech','XLF':'Financials US','XLV':'Healthcare',
        'XLE':'Energy','XLI':'Industrials','XLB':'Materials US',
        'XLU':'Utilities','XLRE':'Real Estate US','XLC':'Comm Services',
        'XLP':'Staples','XLY':'Discretionary',
    },
    # ── SMART BETA / STYLE ───────────────────────────────────
    'smart_beta': {
        'IVW':'S&P Growth','IVE':'S&P Value','USMV':'Min Vol USA',
        'QUAL':'Quality USA','MTUM':'Momentum USA','VIG':'Dividend Growth',
        'DVY':'High Div','SPLV':'Low Vol','SPHB':'High Beta',
        'VLUE':'Enhanced Value',
    },
    # ── EUROPEAN ─────────────────────────────────────────────
    'eu_equity': {
        'EZU':'MSCI Eurozone','EWG':'Germany','EWQ':'France',
        'EWI':'Italy','EWP':'Spain','EWN':'Netherlands',
        'EWD':'Sweden','EWL':'Switzerland','EWU':'UK',
        'IEUR':'iShares Europe','VGK':'Vanguard Europe',
    },
    # ── EMERGING MARKETS ─────────────────────────────────────
    'em_equity': {
        'EEM':'EM Broad','FXI':'China Large','MCHI':'MSCI China',
        'EWZ':'Brazil','EWY':'Korea','EWT':'Taiwan',
        'INDA':'India','EZA':'South Africa','TUR':'Turkey',
        'FM':'Frontier Mkts','VWO':'Vanguard EM',
    },
    # ── ALTERNATIVES ─────────────────────────────────────────
    'alternatives': {
        'GLD':'Gold','SLV':'Silver','PDBC':'Commodities',
        'VNQ':'US REITs','VNQI':'Global REITs','IGF':'Infrastructure',
        'PSP':'Listed PE','USO':'WTI Oil','UNG':'Nat Gas',
        'DJP':'Bloomberg Comm',
    },
    # ── FIXED INCOME ─────────────────────────────────────────
    'fixed_income': {
        'LQD':'IG Corp','HYG':'HY Corp','EMB':'EM Bond USD',
        'EMLC':'EM Local','TIP':'TIPS','TLT':'Long Duration',
        'SHY':'Short Duration','IEF':'Interm Duration',
        'BKLN':'Senior Loans','MBB':'MBS',
        'BWX':'Intl Treasury',
    },
    # ── THEMATIC / MEGATREND ─────────────────────────────────
    'thematic': {
        'BOTZ':'Robotics/AI','ICLN':'Clean Energy',
        'HACK':'Cybersecurity','PHO':'Water',
        'MOO':'Agribusiness','BLOK':'Blockchain',
        'ARKK':'Disruptive Tech','IBB':'Biotech',
    },
}

ALL_ETF_TICKERS = list({t for cat in ETF_UNIVERSE.values() for t in cat})
TICKER_TO_CAT  = {t: cat for cat, items in ETF_UNIVERSE.items() for t in items}
TICKER_TO_NAME = {t: n   for items in ETF_UNIVERSE.values()    for t, n in items.items()}

print(f'🎯 ETF Universe: {len(ALL_ETF_TICKERS)} tickers in {len(ETF_UNIVERSE)} categories')
for cat, items in ETF_UNIVERSE.items():
    print(f'   {cat:<20} {len(items)} ETFs')


🎯 ETF Universe: 72 tickers in 7 categories
   us_sector            11 ETFs
   smart_beta           10 ETFs
   eu_equity            11 ETFs
   em_equity            11 ETFs
   alternatives         10 ETFs
   fixed_income         11 ETFs
   thematic             8 ETFs


In [88]:
# ============================================================
# CELL 4 — Price download from Yahoo Finance
# ============================================================

port_tickers = list(PORTFOLIO.keys())
all_dl = list(set(port_tickers + ALL_ETF_TICKERS + [BENCHMARK]))

print(f'📥 Downloading {len(all_dl)} tickers from Yahoo Finance...')
raw = yf.download(all_dl, start=START_DATE, end=END_DATE,
                  auto_adjust=True, progress=True)
prices = raw['Close'].copy()
if isinstance(prices, pd.Series): prices = prices.to_frame(all_dl[0])

prices = prices.dropna(axis=1, thresh=int(len(prices)*0.80))
prices = prices.ffill().bfill()
prices_r = prices.resample(FREQUENCY).last()
returns  = prices_r.pct_change().dropna()

# Weighted portfolio
valid_pt = [t for t in port_tickers if t in returns.columns]
w_arr    = np.array([PORTFOLIO[t]['weight'] for t in valid_pt])
w_arr   /= w_arr.sum()
returns['PORTFOLIO'] = returns[valid_pt].values @ w_arr

if BENCHMARK in returns.columns:
    returns.rename(columns={BENCHMARK: 'BENCHMARK'}, inplace=True)

freq_mult = {'D':252,'W':52,'M':12}.get(FREQUENCY, 52)
available_etf = [t for t in ALL_ETF_TICKERS if t in returns.columns]

print(f'\n✅ {len(returns)} periods | {len(available_etf)} ETFs available')
ann_ret = (1+returns['PORTFOLIO'].mean())**freq_mult - 1
ann_vol = returns['PORTFOLIO'].std() * freq_mult**0.5
print(f'   Ann. return: {ann_ret:.2%} | Ann. vol: {ann_vol:.2%} | Sharpe: {ann_ret/ann_vol:.3f}')


📥 Downloading 76 tickers from Yahoo Finance...


[*********************100%***********************]  76 of 76 completed




✅ 330 periods | 71 ETFs available
   Ann. return: 17.31% | Ann. vol: 19.32% | Sharpe: 0.896


In [89]:
# ============================================================
# CELL 5 — Multi-regional factor download
# ============================================================

def download_ff_regional(region, freq):
    """
    Download Fama-French factors for the appropriate region.
    Available regions: 'US', 'EU' (Europe), 'EM' (Emerging), 'Global'
    """
    FF_URLS = {
        'US': {
            'ff5': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip',
            'mom': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip',
        },
        'EU': {
            'ff5': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Europe_5_Factors_daily_CSV.zip',
            'mom': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Europe_Mom_Factor_daily_CSV.zip',
        },
        'EM': {
            'ff5': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Emerging_5_Factors_daily_CSV.zip',
            'mom': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Emerging_Mom_Factor_daily_CSV.zip',
        },
        'Global': {
            'ff5': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Developed_5_Factors_daily_CSV.zip',
            'mom': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Developed_Mom_Factor_daily_CSV.zip',
        },
    }
    urls = FF_URLS.get(region, FF_URLS['US'])
    dfs = {}
    for key, url in urls.items():
        try:
            r = requests.get(url, timeout=30)
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                fname = [f for f in z.namelist() if f.upper().endswith('.CSV')][0]
                txt   = z.open(fname).read().decode('utf-8', errors='ignore')
            lines = txt.split('\n')
            start = next(i for i,l in enumerate(lines) if l.strip() and l.strip()[0].isdigit())
            end   = next((i for i,l in enumerate(lines[start:], start)
                          if l.strip() and not l.strip()[0].isdigit() and i>start+20), len(lines))
            df = pd.read_csv(io.StringIO('\n'.join(lines[start:end])),
                             header=None, index_col=0, na_values=['-99.99','-999'])
            df.index = pd.to_datetime(df.index.astype(str), format='%Y%m%d', errors='coerce')
            df = df[df.index.notna()] / 100.0
            if key == 'ff5':
                df.columns = ['Mkt-RF','SMB','HML','RMW','CMA','RF'][:df.shape[1]]
            else:
                df.columns = ['Mom'] + [f'x{i}' for i in range(df.shape[1]-1)]
                df = df[['Mom']]
            df = (1+df).resample(freq).prod() - 1
            df = df[df.index >= pd.Timestamp(START_DATE)]
            dfs[key] = df
            print(f'   ✅ FF-{region}-{key}: {len(df)} periods')
        except Exception as e:
            print(f'   ⚠️  FF-{region}-{key}: fallback ({e})')
            dfs[key] = None
    return dfs


def download_macro_regional(region, start, end, freq):
    """Download macro factors by region (USD=FRED, EUR=FRED+ECB proxy, EM=FRED)."""
    # Common FRED series
    base_series = {
        'TB3MS':'RF_3m','GS10':'Rate10Y_US','GS2':'Rate2Y_US',
        'BAMLC0A0CM':'IG_OAS','BAMLH0A0HYM2':'HY_OAS',
        'CPIAUCSL':'CPI_US','DTWEXBGS':'USD_idx',
        'DCOILWTICO':'Oil_WTI','VIXCLS':'VIX',
    }
    # Additional EU series
    eu_series  = {'IRLTLT01EZM156N':'Rate10Y_EU', 'IRSTCI01EZM156N':'Rate3M_EU',
                  'CP0000EZ17M086NEST':'CPI_EU', 'DEXUSEU':'EUR_USD'}
    # Additional EM series
    em_series  = {'DEXBZUS':'BRL_USD','DEXCHUS':'CNY_USD',
                  'BAMLHE00EHY0EY':'EM_HY_OAS','BAMLEM1RAAA2ACRPIOAS':'EM_IG_OAS'}

    all_series = dict(base_series)
    if region in ('EU','Global'):  all_series.update(eu_series)
    if region in ('EM','Global'):  all_series.update(em_series)

    dfs = []
    for code, col in all_series.items():
        try:
            s = web.DataReader(code,'fred',start,end)[code].rename(col)
            dfs.append(s)
            print(f'   ✅ FRED {code} → {col}')
        except Exception as e:
            print(f'   ⚠️  {code}: {e}')
    return pd.concat(dfs, axis=1) if dfs else pd.DataFrame()


print(f'📥 Downloading FF5 factors for region [{model_type}]...')
ff_dfs    = download_ff_regional(model_type, FREQUENCY)

print(f'\n📥 Downloading FRED macro [{model_type}]...')
macro_raw = download_macro_regional(model_type, START_DATE, END_DATE, FREQUENCY)

print('\n✅ Factor download complete.')


📥 Downloading FF5 factors for region [US]...
   ✅ FF-US-ff5: 322 periods
   ✅ FF-US-ff5: 322 periods
   ✅ FF-US-mom: 322 periods

📥 Downloading FRED macro [US]...
   ✅ FRED TB3MS → RF_3m
   ✅ FF-US-mom: 322 periods

📥 Downloading FRED macro [US]...
   ✅ FRED TB3MS → RF_3m
   ✅ FRED GS10 → Rate10Y_US
   ✅ FRED GS2 → Rate2Y_US
   ✅ FRED GS10 → Rate10Y_US
   ✅ FRED GS2 → Rate2Y_US
   ✅ FRED BAMLC0A0CM → IG_OAS
   ✅ FRED BAMLC0A0CM → IG_OAS
   ✅ FRED BAMLH0A0HYM2 → HY_OAS
   ✅ FRED BAMLH0A0HYM2 → HY_OAS
   ✅ FRED CPIAUCSL → CPI_US
   ✅ FRED CPIAUCSL → CPI_US
   ✅ FRED DTWEXBGS → USD_idx
   ✅ FRED DTWEXBGS → USD_idx
   ✅ FRED DCOILWTICO → Oil_WTI
   ✅ FRED DCOILWTICO → Oil_WTI
   ✅ FRED VIXCLS → VIX

✅ Factor download complete.
   ✅ FRED VIXCLS → VIX

✅ Factor download complete.


In [90]:
# ============================================================
# CELL 6 — Build base factor panel
# ============================================================
import numpy as np
import statsmodels.api as sm

# (Assumes returns, idx, macro_raw, ff5, mom, model_type, FREQUENCY,
#  BENCHMARK, TICKER_TO_NAME, TICKER_TO_CAT, region_weights are already defined)

base = pd.DataFrame(index=idx)

if ff5 is not None and not ff5.empty:
    ff_reindexed = ff5.reindex(idx, method='nearest', tolerance=pd.Timedelta('35d'))
    for col in ['Mkt-RF','SMB','HML','RMW','CMA','RF']:
        if col in ff_reindexed.columns:
            base[col] = ff_reindexed[col]
else:
    np.random.seed(42)
    bench = returns['BENCHMARK'].values if 'BENCHMARK' in returns else np.random.normal(0.002,0.02,len(idx))
    base['Mkt-RF'] = bench - 0.0005
    for f,mu,sg in [('SMB',0.0002,0.007),('HML',0.0001,0.007),
                    ('RMW',0.0002,0.006),('CMA',0.0001,0.005)]:
        base[f] = np.random.normal(mu,sg,len(idx))
    base['RF'] = 0.0001

if mom is not None and 'Mom' in mom.columns:
    base['Mom'] = mom['Mom'].reindex(idx, method='nearest', tolerance=pd.Timedelta('35d'))
else:
    base['Mom'] = np.random.normal(0.0003,0.010,len(idx))

# Macro factors
if not macro_raw.empty:
    m = macro_raw.resample(FREQUENCY).last().reindex(idx, method='nearest', tolerance=pd.Timedelta('14d'))
    # US factors
    if 'Rate10Y_US' in m and 'Rate2Y_US' in m:
        base['TermSpread_US'] = (m['Rate10Y_US'] - m['Rate2Y_US'])/100
    if 'CPI_US' in m:     base['CPI_US']      = m['CPI_US'].pct_change()
    if 'USD_idx' in m:    base['USD_ret']      = m['USD_idx'].pct_change()
    if 'Oil_WTI' in m:    base['Oil_ret']      = m['Oil_WTI'].pct_change()
    if 'VIX' in m:        base['VIX_chg']      = m['VIX'].pct_change()
    if 'IG_OAS' in m:     base['CreditSpread'] = m['IG_OAS'].pct_change()
    if 'HY_OAS' in m and 'IG_OAS' in m:
        base['DefaultSpread'] = (m['HY_OAS'] - m['IG_OAS']).pct_change()
    if 'Rate10Y_US' in m: base['Duration_US']  = -m['Rate10Y_US'].diff()/100
    # EU factors
    if 'Rate10Y_EU' in m and 'Rate3M_EU' in m:
        base['TermSpread_EU'] = (m['Rate10Y_EU'] - m['Rate3M_EU'])/100
    if 'CPI_EU' in m:     base['CPI_EU']       = m['CPI_EU'].pct_change()
    if 'EUR_USD' in m:    base['EUR_USD_ret']   = m['EUR_USD'].pct_change()
    if 'Rate10Y_EU' in m: base['Duration_EU']   = -m['Rate10Y_EU'].diff()/100
    # EM factors
    if 'EM_HY_OAS' in m:  base['EM_CreditSpread'] = m['EM_HY_OAS'].pct_change()
    if 'BRL_USD' in m:    base['BRL_ret']          = m['BRL_USD'].pct_change()
    if 'CNY_USD' in m:    base['CNY_ret']           = m['CNY_USD'].pct_change()

base.replace([np.inf,-np.inf], np.nan, inplace=True)
base.ffill(inplace=True)
base.bfill(inplace=True)

# ── Regional orthogonal market factors ─────────────────────────────────────
# Download EFA (Developed ex-US) and EEM (Emerging Markets), then orthogonalise
# against Mkt-RF to isolate pure regional beta beyond US market.
# Resulting columns  Mkt_EU_orth  and  Mkt_EM_orth  are by construction
# uncorrelated with Mkt-RF (residuals from Mkt_i ~ α + β·Mkt-RF).
_rf_tmp = base['RF'].fillna(0) if 'RF' in base.columns else pd.Series(1e-4, index=base.index)
_mkt    = base['Mkt-RF'].copy() if 'Mkt-RF' in base.columns else None
_interval_map = {'W': '1wk', 'M': '1mo', 'ME': '1mo', 'WE': '1wk'}
_yf_interval  = _interval_map.get(FREQUENCY, '1wk')

if _mkt is not None:
    print('\n📥 Downloading regional market benchmarks for Mkt-RF decomposition...')
    for _col, _tk in [('Mkt_EU_orth', 'EFA'), ('Mkt_EM_orth', 'EEM')]:
        try:
            _px = yf.download(_tk, start=START_DATE, end=END_DATE,
                              interval=_yf_interval, auto_adjust=True, progress=False)
            _px = _px['Close'].squeeze() if 'Close' in _px else _px.squeeze()
            _ret = _px.resample(FREQUENCY).last().pct_change().dropna()
            _ret.index = pd.to_datetime(_ret.index).tz_localize(None)
            _exc = (_ret - _rf_tmp.reindex(_ret.index).fillna(0)).reindex(base.index)
            _aln = pd.concat([_exc.rename('y'), _mkt.rename('x')], axis=1).dropna()
            if len(_aln) >= 30:
                _ols = sm.OLS(_aln['y'], sm.add_constant(_aln['x'])).fit()
                _resid = _ols.resid.reindex(base.index)
                base[_col] = _resid
                base[_col].ffill(inplace=True); base[_col].bfill(inplace=True)
                _corr = base[_col].corr(base['Mkt-RF'])
                print(f'   ✅ {_col} ({_tk}): R²(vs Mkt-RF)={_ols.rsquared:.3f}  '
                      f'residual corr≈{_corr:.3f}  obs={len(_aln)}')
            else:
                print(f'   ⚠️  {_col} ({_tk}): only {len(_aln)} aligned obs — skipped')
        except Exception as _e:
            print(f'   ⚠️  {_col} ({_tk}): {_e}')

print(f'\n✅ Base factor panel [{model_type}]: {base.shape[1]} factors')
print('   Factors:', list(base.columns))

# Excess return
rf_s = base['RF'].reindex(idx).fillna(0.0001) if 'RF' in base.columns else pd.Series(0.0001, index=idx)
excess_ret = returns['PORTFOLIO'] - rf_s



📥 Downloading regional market benchmarks for Mkt-RF decomposition...
   ✅ Mkt_EU_orth (EFA): R²(vs Mkt-RF)=0.290  residual corr≈-0.000  obs=330
   ✅ Mkt_EU_orth (EFA): R²(vs Mkt-RF)=0.290  residual corr≈-0.000  obs=330
   ✅ Mkt_EM_orth (EEM): R²(vs Mkt-RF)=0.244  residual corr≈-0.000  obs=330

✅ Base factor panel [US]: 17 factors
   Factors: ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF', 'Mom', 'TermSpread_US', 'CPI_US', 'USD_ret', 'Oil_ret', 'VIX_chg', 'CreditSpread', 'DefaultSpread', 'Duration_US', 'Mkt_EU_orth', 'Mkt_EM_orth']
   ✅ Mkt_EM_orth (EEM): R²(vs Mkt-RF)=0.244  residual corr≈-0.000  obs=330

✅ Base factor panel [US]: 17 factors
   Factors: ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF', 'Mom', 'TermSpread_US', 'CPI_US', 'USD_ret', 'Oil_ret', 'VIX_chg', 'CreditSpread', 'DefaultSpread', 'Duration_US', 'Mkt_EU_orth', 'Mkt_EM_orth']


In [91]:
# ============================================================
# CELL 7 — ETF orthogonalisation & full regression
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor

# ── Human-readable names for every factor code ──────────────────────────────
FACTOR_LABELS = {
    # Fama-French market & style
    'Mkt-RF':           'Market Risk Premium (US)',
    'Mkt_EU_orth':      'European Market Premium (ex-US, orthogonal)',
    'Mkt_EM_orth':      'Emerging Market Premium (orthogonal)',
    'SMB':              'Size Premium — Small Minus Big',
    'HML':              'Value Premium — High Minus Low Book-to-Market',
    'RMW':              'Profitability Premium — Robust Minus Weak',
    'CMA':              'Investment Premium — Conservative Minus Aggressive',
    'Mom':              'Momentum Factor (12-1 month)',
    # Macro — rates & inflation
    'TermSpread_US':    'US Yield Curve Slope (10Y − 2Y)',
    'TermSpread_EU':    'EUR Yield Curve Slope (10Y − 3M)',
    'Duration_US':      'US Duration Sensitivity (−Δ10Y yield)',
    'Duration_EU':      'EUR Duration Sensitivity (−Δ10Y yield)',
    'CPI_US':           'US Inflation Surprise (CPI mom)',
    'CPI_EU':           'EUR Inflation Surprise (CPI mom)',
    # Macro — credit & default
    'CreditSpread':     'Investment Grade Credit Spread (IG OAS)',
    'DefaultSpread':    'Default Risk Premium (HY − IG spread)',
    'EM_CreditSpread':  'Emerging Market Credit Spread (EM HY OAS)',
    # Macro — FX & commodities
    'USD_ret':          'US Dollar Index Return',
    'EUR_USD_ret':      'EUR/USD Exchange Rate Return',
    'BRL_ret':          'Brazilian Real Return (BRL/USD)',
    'CNY_ret':          'Chinese Yuan Return (CNY/USD)',
    'Oil_ret':          'Crude Oil Return (WTI)',
    # Macro — volatility
    'VIX_chg':          'Equity Volatility Change (VIX % chg)',
}

def factor_display(code, is_etf=False, ticker_name=''):
    """Return a clean, full-english display label for a factor code."""
    if is_etf:
        return f"{code}  [{ticker_name}]"
    return FACTOR_LABELS.get(code, code)


def build_full_model(returns, base, available_etf, excess_ret, max_factors=25,
                     mode='absolute'):
    """
    mode='absolute' : y = portfolio excess return over RF  (total risk attribution)
    mode='active'   : y = portfolio return − benchmark     (active / TE attribution)
    """
    # Include regional orthogonal market factors if present in base
    orth_cols = [c for c in [
        'Mkt-RF', 'Mkt_EU_orth', 'Mkt_EM_orth',
        'SMB', 'HML', 'RMW', 'CMA', 'Mom'
    ] if c in base.columns]

    # Build common index: use all return dates where base factors are available
    base_reindexed = base.reindex(returns.index).ffill().bfill()
    common = returns.index.intersection(base_reindexed.dropna(subset=orth_cols).index)

    ret_s = returns.loc[common]

    # Clean base_s: replace inf, fill any residual NaNs, fallback to 0
    base_s = (base_reindexed.loc[common, orth_cols]
              .replace([np.inf, -np.inf], np.nan)
              .ffill().bfill().fillna(0))

    X_base = sm.add_constant(base_s)

    # ── Choose dependent variable based on mode ──────────────
    if mode == 'active' and 'BENCHMARK' in returns.columns:
        exc_s    = (returns['PORTFOLIO'] - returns['BENCHMARK']).reindex(common).dropna()
        _y_label = 'Active return (Portfolio − Benchmark)'
    else:
        exc_s    = excess_ret.reindex(common).dropna()
        _y_label = 'Excess return (Portfolio − Risk-Free Rate)'

    # Align y and X jointly before OLS (eliminates MissingDataError)
    reg0    = pd.concat([exc_s.rename('y'), X_base], axis=1).dropna()
    r2_base = sm.OLS(reg0['y'], reg0.drop(columns='y')).fit().rsquared

    etf_resid = {}
    etf_stats = []

    for ticker in available_etf:
        if ticker not in ret_s.columns:
            continue
        er = ret_s[ticker].dropna()

        c2_df = pd.concat([er.rename('er'), X_base], axis=1).dropna()
        if len(c2_df) < 30:
            continue
        try:
            ols   = sm.OLS(c2_df['er'], c2_df.drop(columns='er')).fit()
            resid = ols.resid
            rs    = resid.std()
            if rs < 1e-8:
                continue
            rn = resid / rs

            c3_df = pd.concat([exc_s.rename('y'), rn.rename('rn'), base_s], axis=1).dropna()
            if len(c3_df) < 20:
                continue
            X_aug = sm.add_constant(c3_df.drop(columns='y'))
            dr2   = sm.OLS(c3_df['y'], X_aug).fit().rsquared - r2_base

            etf_resid[ticker] = rn.reindex(common)
            etf_stats.append({
                'Ticker':    ticker,
                'Nome':      TICKER_TO_NAME.get(ticker, ticker),
                'Categoria': TICKER_TO_CAT.get(ticker, 'altro'),
                'R2_FF5':    round(ols.rsquared, 4),
                'Alpha_ETF': round(ols.params.get('const', 0) * freq_mult * 100, 3),
                'Resid_Std': round(rs * 100, 4),
                'DeltaR2':   round(dr2, 6),
            })
        except Exception:
            pass

    df_stats = pd.DataFrame(etf_stats).sort_values('DeltaR2', ascending=False) if etf_stats else pd.DataFrame()
    top_etf  = df_stats[df_stats['DeltaR2'] > 0].head(max_factors)['Ticker'].tolist() if not df_stats.empty else []

    etf_panel = pd.DataFrame(
        {f'ETF_{t}': etf_resid[t] for t in top_etf if t in etf_resid},
        index=common
    )

    base_cols   = [c for c in base.columns if c != 'RF']
    base_common = (base.reindex(common)[base_cols]
                   .replace([np.inf, -np.inf], np.nan)
                   .ffill().bfill().fillna(0))
    full_panel  = pd.concat([base_common, etf_panel], axis=1)

    exc_aligned = exc_s.reindex(full_panel.index)
    data_reg    = pd.concat([exc_aligned.rename('y'), full_panel], axis=1).dropna()

    if data_reg.empty:
        raise ValueError(
            f'data_reg is empty after dropna. '
            f'exc_s: {len(exc_s)} obs {exc_s.index.min()} → {exc_s.index.max()}; '
            f'full_panel: {len(full_panel)} obs {full_panel.index.min()} → {full_panel.index.max()}.'
        )

    y = data_reg['y']
    X = sm.add_constant(data_reg.drop(columns='y'))

    ols_full = sm.OLS(y, X).fit(cov_type='HC3')
    ols_ff5  = sm.OLS(y, sm.add_constant(data_reg[orth_cols])).fit()

    # Results table
    factor_names = [c for c in X.columns if c != 'const']
    rows = []
    for f in factor_names:
        beta  = ols_full.params[f]
        tstat = ols_full.tvalues[f]
        pval  = ols_full.pvalues[f]
        ci    = ols_full.conf_int().loc[f]
        contr = beta * data_reg[f].std()
        sig   = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
        is_etf = f.startswith('ETF_')
        tk     = f.replace('ETF_', '')
        rows.append({
            'Fattore':    f,
            'Display':    factor_display(f, is_etf, TICKER_TO_NAME.get(tk, tk)),
            'Categoria':  TICKER_TO_CAT.get(tk, 'macro/fi') if is_etf else (
                          'Mkt-Regional' if f in ('Mkt_EU_orth','Mkt_EM_orth') else 'FF/Macro'),
            'Tipo':       'ETF⊥' if is_etf else ('Mkt⊥' if '⊥' in f or 'orth' in f else 'Base'),
            'Beta':       round(beta, 4),
            'CI_lo':      round(ci[0], 4),
            'CI_hi':      round(ci[1], 4),
            't-stat':     round(tstat, 3),
            'p-value':    round(pval, 4),
            'Sig':        sig,
            'Contributo': round(contr, 6),
        })

    df_res = pd.DataFrame(rows).sort_values('Contributo', key=abs, ascending=False)
    tc = df_res['Contributo'].abs().sum()
    df_res['% var'] = (df_res['Contributo'].abs() / tc * 100).round(2)

    # ── Mkt-RF macro sub-decomposition ──────────────────────────────────────
    mkt_decomp = {}
    _mkt_macro_cols = [c for c in [
        'VIX_chg', 'TermSpread_US', 'CreditSpread', 'DefaultSpread',
        'USD_ret', 'Oil_ret', 'Duration_US', 'TermSpread_EU', 'EM_CreditSpread'
    ] if c in data_reg.columns]

    if 'Mkt-RF' in data_reg.columns and _mkt_macro_cols:
        beta_mkt        = ols_full.params.get('Mkt-RF', 0)
        mkt_contrib_ser = beta_mkt * data_reg['Mkt-RF']
        X_mkt           = sm.add_constant(data_reg[_mkt_macro_cols])
        ols_mkt         = sm.OLS(mkt_contrib_ser, X_mkt).fit()
        sigma_mkt       = mkt_contrib_ser.std()
        for _f in _mkt_macro_cols:
            sub = ols_mkt.params.get(_f, 0) * data_reg[_f].std()
            mkt_decomp[_f] = round(sub / sigma_mkt * 100, 2) if sigma_mkt > 0 else 0
        mkt_decomp['_idiosyncratic_pct'] = round((1 - ols_mkt.rsquared) * 100, 2)
        mkt_decomp['_R2_macro']          = round(ols_mkt.rsquared, 4)
        mkt_decomp['_beta_mkt']          = round(beta_mkt, 4)
        print(f'\n   🔍 Market Risk Premium — macro decomposition [mode={mode}]  '
              f'(β={beta_mkt:+.3f}, macro R²={ols_mkt.rsquared:.3f}):')
        for _f, _v in sorted(mkt_decomp.items(), key=lambda x: abs(x[1]), reverse=True):
            if not _f.startswith('_'):
                print(f'      {FACTOR_LABELS.get(_f,_f):<52}: {_v:+.1f}%')
        print(f'      {"Idiosyncratic (unexplained)":<52}: {mkt_decomp["_idiosyncratic_pct"]:+.1f}%')

    alpha_p   = ols_full.params.get('const', 0)
    alpha_ann = ((1 + alpha_p) ** freq_mult - 1) * 100

    if 'BENCHMARK' in returns.columns:
        ar = returns['PORTFOLIO'] - returns['BENCHMARK']
        te = ar.std() * freq_mult ** 0.5 * 100
        ir = ar.mean() / ar.std() * freq_mult ** 0.5
    else:
        te, ir = np.nan, np.nan

    SUMMARY = {
        'R2_ff5':     round(ols_ff5.rsquared, 4),
        'R2_full':    round(ols_full.rsquared, 4),
        'R2_adj':     round(ols_full.rsquared_adj, 4),
        'DeltaR2':    round(ols_full.rsquared - ols_ff5.rsquared, 4),
        'Alpha_ann':  round(alpha_ann, 3),
        'Alpha_pval': round(ols_full.pvalues.get('const', 1.0), 4),
        'Sharpe':     round(y.mean() / y.std() * freq_mult ** 0.5, 3),
        'TE':         round(te, 2) if not np.isnan(te) else 'N/A',
        'IR':         round(ir, 3) if not np.isnan(ir) else 'N/A',
        'nobs':       int(ols_full.nobs),
        'n_etf':      len(top_etf),
        'n_base':     len(orth_cols),
        'model_type': model_type,
        'mode':       mode,
        'y_label':    _y_label,
        'mkt_decomp': mkt_decomp,
        'orth_cols':  orth_cols,
    }

    return df_res, df_stats, etf_panel, full_panel, data_reg, SUMMARY, ols_full, ols_ff5


# ── Run both models ──────────────────────────────────────────
print('🔢 [Absolute] ETF orthogonalisation + OLS (y = portfolio excess return over RF)...')
df_res, df_etf_stats, etf_panel, full_panel, data_reg, SUMMARY, ols_full, ols_ff5 = \
    build_full_model(returns, base, available_etf, excess_ret, MAX_ETF_FACTORS, mode='absolute')

if 'BENCHMARK' in returns.columns:
    print('\n🔢 [Active]   ETF orthogonalisation + OLS (y = portfolio − benchmark)...')
    df_res_act, _, _, _, data_reg_act, SUMMARY_act, ols_full_act, _ = \
        build_full_model(returns, base, available_etf, excess_ret, MAX_ETF_FACTORS, mode='active')
else:
    print('\n⚠️  No benchmark — active model = absolute model.')
    df_res_act, data_reg_act, SUMMARY_act, ols_full_act = df_res, data_reg, SUMMARY, ols_full

print(f"""
╔══════════════════════════════════╤══════════════════════════════════╗
║  ABSOLUTE (vs Risk-Free Rate)    │  ACTIVE (vs Benchmark)           ║
╠══════════════════════════════════╪══════════════════════════════════╣
║  R² Full:   {SUMMARY['R2_full']:.4f}              │  R² Full:   {SUMMARY_act['R2_full']:.4f}              ║
║  Alpha ann: {SUMMARY['Alpha_ann']:+.2f}%              │  Alpha ann: {SUMMARY_act['Alpha_ann']:+.2f}%              ║
║  TE ann.:   {str(SUMMARY['TE']):<22}│  TE ann.:   {str(SUMMARY_act['TE']):<22}║
╚══════════════════════════════════╧══════════════════════════════════╝
""")
print('── Absolute top factors ──')
display(df_res.head(15)[['Display','Categoria','Tipo','Beta','t-stat','p-value','Sig','% var']]
        .style.format({'Beta':'{:+.4f}','t-stat':'{:+.3f}','p-value':'{:.4f}','% var':'{:.2f}%'})
        .bar(subset=['% var'], color='#3266ad'))
if 'BENCHMARK' in returns.columns:
    print('── Active top factors ──')
    display(df_res_act.head(15)[['Display','Categoria','Tipo','Beta','t-stat','p-value','Sig','% var']]
            .style.format({'Beta':'{:+.4f}','t-stat':'{:+.3f}','p-value':'{:.4f}','% var':'{:.2f}%'})
            .bar(subset=['% var'], color='#f5a623'))


🔢 [Absolute] ETF orthogonalisation + OLS (y = portfolio excess return over RF)...

   🔍 Market Risk Premium — macro decomposition [mode=absolute]  (β=+1.032, macro R²=0.697):
      Equity Volatility Change (VIX % chg)                : -50.1%
      US Dollar Index Return                              : -49.0%
      Default Risk Premium (HY − IG spread)               : -4.3%
      US Yield Curve Slope (10Y − 2Y)                     : +3.2%
      Crude Oil Return (WTI)                              : +2.9%
      Investment Grade Credit Spread (IG OAS)             : -1.7%
      US Duration Sensitivity (−Δ10Y yield)               : +0.0%
      Idiosyncratic (unexplained)                         : +30.3%

🔢 [Active]   ETF orthogonalisation + OLS (y = portfolio − benchmark)...

   🔍 Market Risk Premium — macro decomposition [mode=absolute]  (β=+1.032, macro R²=0.697):
      Equity Volatility Change (VIX % chg)                : -50.1%
      US Dollar Index Return                              : -

,Display,Categoria,Tipo,Beta,t-stat,p-value,Sig,% var
0,Market Risk Premium (US),FF/Macro,Base,+1.0319,+105.515,0.0000,***,55.24%
16,ETF_IVW [S&P Growth],smart_beta,ETF⊥,+0.0044,+24.687,0.0000,***,9.35%
1,Size Premium — Small Minus Big,FF/Macro,Base,-0.4360,-29.644,0.0000,***,8.98%
20,ETF_IVE [S&P Value],smart_beta,ETF⊥,+0.0033,+18.004,0.0000,***,6.97%
14,"European Market Premium (ex-US, orthogonal)",Mkt-Regional,Mkt⊥,-0.0700,-13.287,0.0000,***,3.21%
30,ETF_XLRE [Real Estate US],us_sector,ETF⊥,-0.0010,-1.747,0.0806,*,2.06%
29,ETF_VNQ [US REITs],alternatives,ETF⊥,+0.0008,+1.604,0.1087,,1.78%
4,Investment Premium — Conservative Minus Aggressive,FF/Macro,Base,+0.0765,+7.787,0.0000,***,1.76%
3,Profitability Premium — Robust Minus Weak,FF/Macro,Base,+0.1015,+9.297,0.0000,***,1.72%
15,Emerging Market Premium (orthogonal),Mkt-Regional,Mkt⊥,+0.0336,+10.474,0.0000,***,1.71%


── Active top factors ──


,Display,Categoria,Tipo,Beta,t-stat,p-value,Sig,% var
36,ETF_VWO [Vanguard EM],em_equity,ETF⊥,+0.0009,+1.759,0.0786,*,20.86%
29,ETF_EEM [EM Broad],em_equity,ETF⊥,-0.0009,-1.565,0.1175,,20.72%
24,ETF_XLRE [Real Estate US],us_sector,ETF⊥,-0.0002,-0.589,0.5560,,5.59%
1,Size Premium — Small Minus Big,FF/Macro,Base,+0.0201,+1.452,0.1466,,4.57%
31,ETF_EWY [Korea],em_equity,ETF⊥,+0.0002,+1.427,0.1536,,4.54%
16,ETF_TIP [TIPS],fixed_income,ETF⊥,+0.0002,+1.589,0.1120,,4.31%
28,ETF_VNQ [US REITs],alternatives,ETF⊥,+0.0002,+0.427,0.6691,,3.92%
14,"European Market Premium (ex-US, orthogonal)",Mkt-Regional,Mkt⊥,+0.0067,+1.536,0.1245,,3.40%
9,Crude Oil Return (WTI),FF/Macro,Base,-0.0016,-1.074,0.2828,,3.08%
25,ETF_VLUE [Enhanced Value],smart_beta,ETF⊥,-0.0001,-1.473,0.1408,,2.45%


In [92]:
# ============================================================
# CELL 8 — Risk metrics for the dashboard
# ============================================================

def compute_risk_metrics(returns_series, freq_mult):
    """Compute full risk metrics for the dashboard."""
    r = returns_series.dropna()

    # Cumulative
    cum = (1+r).cumprod()

    # Drawdown
    roll_max = cum.cummax()
    dd       = (cum / roll_max) - 1
    max_dd   = dd.min()

    # VaR & CVaR
    var_95   = np.percentile(r, 5)
    var_99   = np.percentile(r, 1)
    cvar_95  = r[r <= var_95].mean()
    cvar_99  = r[r <= var_99].mean()

    # Annualised
    ann_ret  = (1+r.mean())**freq_mult - 1
    ann_vol  = r.std() * freq_mult**0.5
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else np.nan

    # Sortino
    neg_r    = r[r < 0]
    sortino  = ann_ret / (neg_r.std()*freq_mult**0.5) if len(neg_r)>0 else np.nan

    # Calmar
    calmar   = ann_ret / abs(max_dd) if max_dd < 0 else np.nan

    # Skew, Kurt
    skew_    = stats.skew(r)
    kurt_    = stats.kurtosis(r)

    # Rolling Sharpe (26-period window)
    window_r = min(26, len(r)//3)
    roll_sr  = r.rolling(window_r).apply(
        lambda x: (x.mean()/x.std())*freq_mult**0.5 if x.std()>0 else np.nan)

    # Rolling Vol
    roll_vol = r.rolling(window_r).std() * freq_mult**0.5 * 100

    # Monthly returns
    monthly  = r.resample('ME').apply(lambda x:(1+x).prod()-1) * 100

    return {
        'cum':       cum, 'dd': dd, 'roll_sr': roll_sr, 'roll_vol': roll_vol,
        'monthly':   monthly,
        'ann_ret':   ann_ret,   'ann_vol':  ann_vol,   'sharpe':   sharpe,
        'sortino':   sortino,   'calmar':   calmar,    'max_dd':   max_dd,
        'var_95':    var_95,    'var_99':   var_99,
        'cvar_95':   cvar_95,   'cvar_99':  cvar_99,
        'skew':      skew_,     'kurt':     kurt_,
    }

metrics_port  = compute_risk_metrics(returns['PORTFOLIO'], freq_mult)
metrics_bench = compute_risk_metrics(returns['BENCHMARK'], freq_mult) if 'BENCHMARK' in returns else None

# Regional breakdown
regional_returns = {}
for region in set(v['region'] for v in PORTFOLIO.values()):
    reg_tickers = [t for t,v in PORTFOLIO.items() if v['region']==region and t in returns.columns]
    if not reg_tickers: continue
    reg_weights = np.array([PORTFOLIO[t]['weight'] for t in reg_tickers])
    reg_weights /= reg_weights.sum()
    regional_returns[region] = (returns[reg_tickers].values @ reg_weights)
    regional_returns[region] = pd.Series(regional_returns[region], index=returns.index)

print('✅ Risk metrics computed.')
print(f'   Sharpe: {metrics_port["sharpe"]:.3f} | Sortino: {metrics_port["sortino"]:.3f} | Calmar: {metrics_port["calmar"]:.3f}')
print(f'   Max DD: {metrics_port["max_dd"]:.2%} | VaR 95%: {metrics_port["var_95"]:.2%} | CVaR 95%: {metrics_port["cvar_95"]:.2%}')


✅ Risk metrics computed.
   Sharpe: 0.896 | Sortino: 1.169 | Calmar: 0.544
   Max DD: -31.83% | VaR 95%: -3.28% | CVaR 95%: -6.06%


In [93]:
# ============================================================
# CELL 9 — Multi-sheet Excel export
# ============================================================

OUTPUT_DIR = './attribution_reports_dashboard'
os.makedirs(OUTPUT_DIR, exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M')
xlsx_path = f'{OUTPUT_DIR}/attribution_multiregional_{ts}.xlsx'

with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:

    # 1 — Factor Attribution
    df_res[['Display','Categoria','Tipo','Beta','CI_lo','CI_hi',
            't-stat','p-value','Sig','Contributo','% var']]\
        .to_excel(writer, sheet_name='Factor_Attribution', index=False)

    # 2 — ETF Universe Stats
    df_etf_stats.to_excel(writer, sheet_name='ETF_Universe', index=False)

    # 3 — Risk Metrics
    risk_rows = [
        ('Annualised Return',           f"{metrics_port['ann_ret']:.2%}"),
        ('Annualised Volatility',       f"{metrics_port['ann_vol']:.2%}"),
        ('Sharpe Ratio',                f"{metrics_port['sharpe']:.3f}"),
        ('Sortino Ratio',               f"{metrics_port['sortino']:.3f}"),
        ('Calmar Ratio',                f"{metrics_port['calmar']:.3f}"),
        ('Max Drawdown',                f"{metrics_port['max_dd']:.2%}"),
        ('VaR 95% (periodic)',          f"{metrics_port['var_95']:.2%}"),
        ('VaR 99% (periodic)',          f"{metrics_port['var_99']:.2%}"),
        ('CVaR 95%',                    f"{metrics_port['cvar_95']:.2%}"),
        ('CVaR 99%',                    f"{metrics_port['cvar_99']:.2%}"),
        ('Skewness',                    f"{metrics_port['skew']:.3f}"),
        ('Excess Kurtosis',             f"{metrics_port['kurt']:.3f}"),
        ('Annualised Alpha',            f"{SUMMARY['Alpha_ann']:+.2f}%"),
        ('Alpha p-value',               f"{SUMMARY['Alpha_pval']:.4f}"),
        ('R² FF5-base',                 f"{SUMMARY['R2_ff5']:.4f}"),
        ('R² Full model',               f"{SUMMARY['R2_full']:.4f}"),
        ('ΔR² from ETFs',               f"{SUMMARY['DeltaR2']:.4f}"),
        ('Tracking Error',              f"{SUMMARY['TE']}%"),
        ('Information Ratio',           f"{SUMMARY['IR']}"),
        ('Regional model',              SUMMARY['model_type']),
    ]
    pd.DataFrame(risk_rows, columns=['Metric','Value'])\
        .to_excel(writer, sheet_name='Risk_Metrics', index=False)

    # 4 — Returns
    returns[['PORTFOLIO']+(['BENCHMARK'] if 'BENCHMARK' in returns else [])]\
        .to_excel(writer, sheet_name='Returns')

    # 5 — Monthly Heatmap
    mret = returns['PORTFOLIO'].resample('ME').apply(lambda x:(1+x).prod()-1)*100
    mdf  = mret.to_frame('ret')
    mdf['year'] = mdf.index.year; mdf['month'] = mdf.index.month
    pivot = mdf.pivot(index='year', columns='month', values='ret')
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    pivot.to_excel(writer, sheet_name='Monthly_Heatmap')

    # 6 — ETF Factor Residuals
    etf_panel.to_excel(writer, sheet_name='ETF_Factor_Residuals')

    # 7 — Portfolio
    pd.DataFrame([{'Ticker':t,'Name':v['name'],'Weight':f"{v['weight']:.2%}",
                   'Asset class':v['asset_class'],'Region':v.get('region','')}
                  for t,v in PORTFOLIO.items()])\
        .to_excel(writer, sheet_name='Portfolio', index=False)

    # 8 — Regional Breakdown
    reg_rows = [{'Region':r,'Weight':f"{w:.1%}"} for r,w in region_weights.items()]
    pd.DataFrame(reg_rows).to_excel(writer, sheet_name='Regional_Breakdown', index=False)

print(f'✅ Excel exported (8 sheets): {xlsx_path}')


✅ Excel exported (8 sheets): ./attribution_reports_dashboard/attribution_multiregional_20260503_1535.xlsx


In [94]:
# ============================================================
# CELL 10 — INTERACTIVE DASH DASHBOARD
# ============================================================

import dash
from dash import dcc, html, Input, Output, dash_table
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Colour palette ───────────────────────────────────────────
COLORS = {
    'bg':       '#0f1117', 'card':    '#1a1d2e', 'card2':   '#232840',
    'border':   '#2d3154', 'text':    '#e8eaf6', 'sub':     '#8892b0',
    'blue':     '#4f8ef7', 'green':   '#43d9a2', 'red':     '#f47174',
    'orange':   '#f5a623', 'purple':  '#a78bfa', 'teal':    '#22d3ee',
    'gold':     '#fbbf24',
}
CAT_PAL = {
    'FF/Macro':     '#4f8ef7', 'us_sector':    '#43d9a2',
    'smart_beta':   '#22d3ee', 'eu_equity':    '#f5a623',
    'em_equity':    '#f47174', 'alternatives': '#a78bfa',
    'fixed_income': '#fbbf24', 'thematic':     '#fb923c',
    'macro/fi':     '#818cf8', 'Mkt-Regional': '#fb923c',
}

# ── Helpers ──────────────────────────────────────────────────
def card(title, content, color=None):
    border_color = color or COLORS['border']
    return html.Div([
        html.Div(title, style={'fontSize':'11px','color':COLORS['sub'],
                               'textTransform':'uppercase','letterSpacing':'1px','marginBottom':'6px'}),
        html.Div(content, style={'fontSize':'22px','fontWeight':'600','color':color or COLORS['text']})
    ], style={'background':COLORS['card'],'border':f'1px solid {border_color}20',
              'borderRadius':'10px','padding':'16px 20px'})

def section_title(t):
    return html.Div(t, style={'fontSize':'13px','fontWeight':'600','color':COLORS['sub'],
                               'textTransform':'uppercase','letterSpacing':'1.5px',
                               'marginBottom':'12px','marginTop':'8px',
                               'borderLeft':f'3px solid {COLORS["blue"]}','paddingLeft':'10px'})

# ── Pre-compute chart data ───────────────────────────────────
cum_port  = (1+returns['PORTFOLIO']).cumprod()
cum_bench = (1+returns['BENCHMARK']).cumprod() if 'BENCHMARK' in returns else None
dd_series = metrics_port['dd']

mret  = returns['PORTFOLIO'].resample('ME').apply(lambda x:(1+x).prod()-1)*100
mdf2  = mret.to_frame('ret')
mdf2['year'] = mdf2.index.year; mdf2['month'] = mdf2.index.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot_heat  = mdf2.pivot(index='year', columns='month', values='ret').fillna(0)

# Pre-compute factor contribution tables for both modes
top_factors_abs = df_res.head(20).sort_values('Contributo')       # absolute
top_factors_act = df_res_act.head(20).sort_values('Contributo')   # active
top_etf_plot    = df_etf_stats.sort_values('DeltaR2', ascending=False).head(20)
_has_benchmark  = 'BENCHMARK' in returns.columns

# Regional cumulative
reg_colors = {'US':COLORS['blue'],'EU':COLORS['orange'],
              'EM':COLORS['red'],'Global':COLORS['green']}

# ── APP LAYOUT ───────────────────────────────────────────────
app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.CYBORG],
    suppress_callback_exceptions=True,
    title='Factor Attribution Dashboard'
)

GLOBAL_STYLE = {'backgroundColor':COLORS['bg'],'minHeight':'100vh',
                'fontFamily':'Inter, system-ui, sans-serif','color':COLORS['text']}

# ── HEADER ───────────────────────────────────────────────────
header = html.Div([
    html.Div([
        html.Div([
            html.Span('📊 ', style={'fontSize':'22px'}),
            html.Span('Factor Attribution Engine', style={'fontSize':'20px','fontWeight':'700'}),
            html.Span(f' — {model_type} Model',
                      style={'fontSize':'13px','color':COLORS['blue'],'marginLeft':'10px'}),
        ]),
        html.Div(f"{START_DATE} → {END_DATE}  ·  {FREQUENCY}-frequency  ·  "
                 f"{len(returns)} obs  ·  {SUMMARY['n_etf']} ETF factors",
                 style={'fontSize':'12px','color':COLORS['sub'],'marginTop':'3px'}),
    ], style={'flex':'1'}),
    html.Div([
        html.Div(f"R² {SUMMARY['R2_full']:.3f}",
                 style={'background':f"{COLORS['blue']}20",'border':f"1px solid {COLORS['blue']}40",
                        'borderRadius':'6px','padding':'4px 12px','fontSize':'13px','fontWeight':'600',
                        'color':COLORS['blue'],'marginRight':'8px'}),
        html.Div(f"α {SUMMARY['Alpha_ann']:+.1f}%",
                 style={'background':f"{(COLORS['green'] if SUMMARY['Alpha_ann']>0 else COLORS['red'])}20",
                        'border':f"1px solid {(COLORS['green'] if SUMMARY['Alpha_ann']>0 else COLORS['red'])}40",
                        'borderRadius':'6px','padding':'4px 12px','fontSize':'13px','fontWeight':'600',
                        'color':COLORS['green'] if SUMMARY['Alpha_ann']>0 else COLORS['red']}),
    ], style={'display':'flex','alignItems':'center'})
], style={'display':'flex','justifyContent':'space-between','alignItems':'center',
           'padding':'18px 28px','background':COLORS['card'],
           'borderBottom':f'1px solid {COLORS["border"]}'})

# ── TABS ─────────────────────────────────────────────────────
tabs = dcc.Tabs(id='tabs', value='tab-perf', children=[
    dcc.Tab(label='📈 Performance',  value='tab-perf',    style={'color':COLORS['sub']}, selected_style={'color':COLORS['text'],'borderBottom':f'2px solid {COLORS["blue"]}','background':COLORS['card2']}),
    dcc.Tab(label='🔬 Attribution',  value='tab-attr',    style={'color':COLORS['sub']}, selected_style={'color':COLORS['text'],'borderBottom':f'2px solid {COLORS["blue"]}','background':COLORS['card2']}),
    dcc.Tab(label='⚠️ Risk',         value='tab-risk',    style={'color':COLORS['sub']}, selected_style={'color':COLORS['text'],'borderBottom':f'2px solid {COLORS["blue"]}','background':COLORS['card2']}),
    dcc.Tab(label='🗺️ ETF Map',      value='tab-etf',     style={'color':COLORS['sub']}, selected_style={'color':COLORS['text'],'borderBottom':f'2px solid {COLORS["blue"]}','background':COLORS['card2']}),
    dcc.Tab(label='🌡️ Heatmap',      value='tab-heat',    style={'color':COLORS['sub']}, selected_style={'color':COLORS['text'],'borderBottom':f'2px solid {COLORS["blue"]}','background':COLORS['card2']}),
    dcc.Tab(label='🌍 Regions',      value='tab-regions', style={'color':COLORS['sub']}, selected_style={'color':COLORS['text'],'borderBottom':f'2px solid {COLORS["blue"]}','background':COLORS['card2']}),
], style={'background':COLORS['card'],'border':'none','borderBottom':f'1px solid {COLORS["border"]}'},
   colors={'border':COLORS['card'],'primary':COLORS['blue'],'background':COLORS['card']})

app.layout = html.Div([
    header,
    html.Div([
        # KPI strip
        html.Div([
            card('Ann. Return',    f"{metrics_port['ann_ret']:.2%}", COLORS['green'] if metrics_port['ann_ret']>0 else COLORS['red']),
            card('Ann. Volatility', f"{metrics_port['ann_vol']:.2%}"),
            card('Sharpe Ratio',   f"{metrics_port['sharpe']:.3f}", COLORS['green'] if metrics_port['sharpe']>1 else (COLORS['orange'] if metrics_port['sharpe']>0.5 else COLORS['red'])),
            card('Max Drawdown',   f"{metrics_port['max_dd']:.2%}", COLORS['red']),
            card('VaR 95%',        f"{metrics_port['var_95']:.2%}", COLORS['orange']),
            card('Ann. Alpha',     f"{SUMMARY['Alpha_ann']:+.2f}%", COLORS['green'] if SUMMARY['Alpha_ann']>0 else COLORS['red']),
            card('Tracking Error', f"{SUMMARY['TE']}%"),
            card('Info Ratio',     f"{SUMMARY['IR']}"),
        ], style={'display':'grid','gridTemplateColumns':'repeat(8,1fr)',
                  'gap':'10px','padding':'16px 24px'}),
        tabs,
        # ── Attribution basis toggle (always in DOM, used by callback) ──────
        html.Div([
            html.Span('Attribution basis:',
                      style={'fontSize':'12px','color':COLORS['sub'],
                             'marginRight':'14px','fontWeight':'500','letterSpacing':'0.5px'}),
            dcc.RadioItems(
                id='attr-mode',
                options=[
                    {'label': ' 📊  Absolute  —  port vs RF  (total risk decomposition)',
                     'value': 'absolute'},
                    {'label': ' 🎯  Active  —  port vs benchmark  (tracking-error decomposition)',
                     'value': 'active',
                     'disabled': not _has_benchmark},
                ],
                value='absolute',
                inline=True,
                inputStyle={'marginRight':'5px'},
                labelStyle={'marginRight':'28px','fontSize':'12px',
                            'color':COLORS['text'],'cursor':'pointer'},
            ),
        ], style={
            'display':'flex','alignItems':'center',
            'padding':'9px 24px',
            'background':COLORS['card2'],
            'borderBottom':f'1px solid {COLORS["border"]}',
        }),
        html.Div(id='tab-content', style={'padding':'20px 24px'})
    ])
], style=GLOBAL_STYLE)

# ── CALLBACKS ────────────────────────────────────────────────
@app.callback(
    Output('tab-content','children'),
    Input('tabs','value'),
    Input('attr-mode','value'),
)
def render_tab(tab, attr_mode):
    attr_mode = attr_mode or 'absolute'

    # ── TAB 1: PERFORMANCE ────────────────────────────────────
    if tab == 'tab-perf':
        fig_cum = go.Figure()
        fig_cum.add_trace(go.Scatter(x=cum_port.index, y=cum_port.values,
            name='Portfolio', line=dict(color=COLORS['blue'],width=2.5)))
        if cum_bench is not None:
            fig_cum.add_trace(go.Scatter(x=cum_bench.index, y=cum_bench.values,
                name='Benchmark', line=dict(color=COLORS['sub'],width=1.5,dash='dot')))
            diff = cum_port - cum_bench
            fig_cum.add_trace(go.Scatter(x=cum_port.index, y=cum_bench.values,
                fill=None, mode='lines', line=dict(width=0), showlegend=False))
            fig_cum.add_trace(go.Scatter(x=cum_port.index, y=cum_port.values,
                fill='tonexty', mode='lines', line=dict(width=0),
                fillcolor='rgba(79,142,247,0.10)', name='Excess', showlegend=False))
        fig_cum.update_layout(
            title='Cumulative Performance (base 1.0)',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'], size=11),
            legend=dict(bgcolor='rgba(0,0,0,0)', orientation='h', y=1.1),
            xaxis=dict(gridcolor=COLORS['border']), yaxis=dict(gridcolor=COLORS['border']),
            hovermode='x unified', margin=dict(t=45,b=30,l=50,r=20)
        )

        fig_dd = go.Figure()
        fig_dd.add_trace(go.Scatter(x=dd_series.index, y=dd_series.values*100,
            fill='tozeroy', fillcolor='rgba(244,113,116,0.20)',
            line=dict(color=COLORS['red'],width=1.5), name='Drawdown'))
        fig_dd.update_layout(
            title='Drawdown (%)', paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=11),
            xaxis=dict(gridcolor=COLORS['border']), yaxis=dict(gridcolor=COLORS['border'],tickformat='.1f',ticksuffix='%'),
            hovermode='x unified', margin=dict(t=45,b=30,l=50,r=20)
        )

        fig_vol = go.Figure()
        fig_vol.add_trace(go.Scatter(x=metrics_port['roll_vol'].index,
            y=metrics_port['roll_vol'].values, fill='tozeroy',
            fillcolor='rgba(79,142,247,0.15)',
            line=dict(color=COLORS['blue'],width=1.5), name='Rolling Vol'))
        fig_vol.update_layout(
            title='Rolling Annualised Volatility (%)',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=11),
            xaxis=dict(gridcolor=COLORS['border']),
            yaxis=dict(gridcolor=COLORS['border'],ticksuffix='%'),
            margin=dict(t=45,b=30,l=50,r=20)
        )

        return html.Div([
            dcc.Graph(figure=fig_cum,  style={'marginBottom':'16px'}),
            html.Div([dcc.Graph(figure=fig_dd), dcc.Graph(figure=fig_vol)],
                     style={'display':'grid','gridTemplateColumns':'1fr 1fr','gap':'16px'})
        ])

    # ── TAB 2: ATTRIBUTION ────────────────────────────────────
    elif tab == 'tab-attr':
        # Select result set based on toggle
        _tf    = top_factors_act  if attr_mode == 'active' else top_factors_abs
        _summ  = SUMMARY_act      if attr_mode == 'active' else SUMMARY
        _dr    = data_reg_act     if attr_mode == 'active' else data_reg
        _accent = COLORS['orange'] if attr_mode == 'active' else COLORS['blue']

        if attr_mode == 'active':
            _mode_label  = 'Active (Port − Benchmark)  ·  tracking-error decomposition'
            _contr_axis  = 'β_active × σ_factor  (tracking-error units)'
        else:
            _mode_label  = 'Absolute (Port − RF)  ·  total risk decomposition'
            _contr_axis  = 'β × σ_factor  (total risk units)'

        bar_colors = [CAT_PAL.get(r['Categoria'],COLORS['sub']) for _,r in _tf.iterrows()]
        opacities  = [1.0 if r['p-value']<0.10 else 0.3 for _,r in _tf.iterrows()]

        fig_bar = go.Figure(go.Bar(
            x=_tf['Contributo'], y=_tf['Display'],
            orientation='h',
            marker=dict(color=bar_colors, opacity=opacities,
                        line=dict(color=COLORS['bg'],width=0.5)),
            text=_tf.apply(lambda r: f"{r['Sig']} {r['% var']:.1f}%", axis=1),
            textposition='outside', textfont=dict(size=10, color=COLORS['sub'])
        ))
        fig_bar.add_vline(x=0, line=dict(color=COLORS['sub'],width=1))
        fig_bar.update_layout(
            title=f'Factor contribution — {_mode_label}',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=10), height=520,
            xaxis=dict(gridcolor=COLORS['border'],title=_contr_axis),
            yaxis=dict(gridcolor='rgba(0,0,0,0)',automargin=True),
            margin=dict(t=45,b=30,l=220,r=80)
        )

        # R² donut
        r2_vals   = [_summ['R2_ff5'], _summ['DeltaR2'], max(0,1-_summ['R2_full'])]
        r2_labels = ['FF5/Macro/Credit','ETF orthog. factors⊥','Idiosyncratic']
        r2_colors = [_accent, COLORS['green'], COLORS['border']]
        fig_donut = go.Figure(go.Pie(
            values=r2_vals, labels=r2_labels, hole=0.60,
            marker=dict(colors=r2_colors, line=dict(color=COLORS['bg'],width=2)),
            textfont=dict(size=11)
        ))
        fig_donut.add_annotation(text=f"R²<br>{_summ['R2_full']:.3f}",
            x=0.5, y=0.5, showarrow=False, font=dict(size=15,color=COLORS['text'],weight='bold'))
        fig_donut.update_layout(
            title=f'Variance decomposition  ·  {"active" if attr_mode=="active" else "absolute"}',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card'],
            font=dict(color=COLORS['text'],size=11),
            legend=dict(bgcolor='rgba(0,0,0,0)', orientation='v', x=1.0),
            margin=dict(t=45,b=30,l=20,r=20), height=300
        )

        # Beta CI chart (top 10 by contribution)
        df_ci = _tf.sort_values('Contributo', key=abs, ascending=False).head(10)
        fig_ci = go.Figure()
        for _,r in df_ci.iterrows():
            c = COLORS['green'] if r['Beta']>0 else COLORS['red']
            fig_ci.add_trace(go.Scatter(
                x=[r['CI_lo'], r['Beta'], r['CI_hi']], y=[r['Display']]*3,
                mode='lines+markers',
                line=dict(color=c, width=1.5),
                marker=dict(size=[5,10,5], color=[COLORS['sub'],c,COLORS['sub']]),
                showlegend=False
            ))
        fig_ci.add_vline(x=0, line=dict(color=COLORS['sub'],width=1,dash='dot'))
        fig_ci.update_layout(
            title=f'Beta with 95% confidence intervals  ·  {"active" if attr_mode=="active" else "absolute"}',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=10), height=300,
            xaxis=dict(gridcolor=COLORS['border'],
                       title='Active Beta (β_tilt)' if attr_mode=='active' else 'Beta'),
            yaxis=dict(gridcolor='rgba(0,0,0,0)',automargin=True),
            margin=dict(t=45,b=30,l=180,r=20)
        )

        # ── Mkt-RF macro sub-decomposition chart ────────────────
        mkt_d = _summ.get('mkt_decomp', {})
        # Use the same FACTOR_LABELS dict defined in cell 7 (with emoji prefix for chart)
        _emoji_prefix = {
            'VIX_chg':       '📉 ', 'TermSpread_US': '📐 ', 'CreditSpread':   '💳 ',
            'DefaultSpread': '⚠️ ', 'USD_ret':       '💵 ', 'Oil_ret':        '🛢️ ',
            'Duration_US':   '⏳ ', 'TermSpread_EU': '📐 ', 'EM_CreditSpread':'🌍 ',
        }
        _driver_vals   = {k: v for k, v in mkt_d.items() if not k.startswith('_')}
        _idio_pct      = mkt_d.get('_idiosyncratic_pct', 0)
        _r2_macro      = mkt_d.get('_R2_macro', 0)
        _beta_mkt      = mkt_d.get('_beta_mkt', None)

        mkt_decomp_children = []
        if _driver_vals:
            _labels  = [_emoji_prefix.get(k,'') + FACTOR_LABELS.get(k, k) for k in _driver_vals]
            _values  = list(_driver_vals.values())
            _d_colors = [COLORS['green'] if v >= 0 else COLORS['red'] for v in _values]
            fig_mkt = go.Figure()
            fig_mkt.add_trace(go.Bar(
                x=_values, y=_labels, orientation='h',
                marker=dict(color=_d_colors, opacity=0.82,
                            line=dict(color=COLORS['bg'], width=0.5)),
                text=[f'{v:+.1f}%' for v in _values],
                textposition='outside', textfont=dict(size=10, color=COLORS['sub'])
            ))
            fig_mkt.add_trace(go.Bar(
                x=[_idio_pct], y=['🔘 Idiosyncratic (unexplained)'],
                orientation='h',
                marker=dict(color=COLORS['border'], opacity=0.7),
                text=[f'{_idio_pct:.1f}%'], textposition='outside',
                textfont=dict(size=10, color=COLORS['sub']),
                showlegend=False
            ))
            fig_mkt.add_vline(x=0, line=dict(color=COLORS['sub'], width=1))
            _title_beta = f'β={_beta_mkt:+.3f}' if _beta_mkt is not None else ''
            fig_mkt.update_layout(
                title=(f'Mkt-RF macro decomposition  ·  {_title_beta}  ·  macro R²={_r2_macro:.3f}  '
                       f'·  {"active" if attr_mode=="active" else "absolute"}<br>'
                       f'<sup>Each bar = contribution of that macro driver to the market-beta exposure (%σ)</sup>'),
                paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
                font=dict(color=COLORS['text'], size=10), height=340,
                xaxis=dict(gridcolor=COLORS['border'],
                           title='Normalised contribution (% of Mkt-RF σ)', ticksuffix='%'),
                yaxis=dict(gridcolor='rgba(0,0,0,0)', automargin=True),
                barmode='relative',
                margin=dict(t=70, b=30, l=220, r=80)
            )
            mkt_decomp_children = [
                section_title('Mkt-RF decomposition into macro drivers'),
                dcc.Graph(figure=fig_mkt)
            ]
        else:
            mkt_decomp_children = [
                html.Div('Mkt-RF macro decomposition not available (no macro factors in base panel).',
                         style={'color': COLORS['sub'], 'padding': '10px'})
            ]

        return html.Div([
            dcc.Graph(figure=fig_bar),
            html.Div([dcc.Graph(figure=fig_donut), dcc.Graph(figure=fig_ci)],
                     style={'display':'grid','gridTemplateColumns':'1fr 1fr',
                            'gap':'16px','marginTop':'16px'}),
            html.Div(mkt_decomp_children,
                     style={'marginTop':'20px','background':COLORS['card'],
                            'borderRadius':'10px','padding':'16px'})
        ])

    # ── TAB 3: RISK ───────────────────────────────────────────
    elif tab == 'tab-risk':
        fig_sr = go.Figure()
        fig_sr.add_trace(go.Scatter(
            x=metrics_port['roll_sr'].index, y=metrics_port['roll_sr'].values,
            fill='tozeroy', fillcolor='rgba(67,217,162,0.12)',
            line=dict(color=COLORS['green'],width=2), name='Rolling Sharpe'))
        fig_sr.add_hline(y=0, line=dict(color=COLORS['sub'],width=1,dash='dot'))
        fig_sr.add_hline(y=1, line=dict(color=COLORS['green'],width=0.8,dash='dash'),
                         annotation_text='Sharpe=1', annotation_position='right')
        fig_sr.update_layout(
            title='Rolling Sharpe ratio (26-period window)',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=11),
            xaxis=dict(gridcolor=COLORS['border']),
            yaxis=dict(gridcolor=COLORS['border']),
            hovermode='x unified', margin=dict(t=45,b=30,l=50,r=20)
        )

        r_vals = returns['PORTFOLIO'].dropna()*100
        fig_dist = go.Figure()
        fig_dist.add_trace(go.Histogram(
            x=r_vals, nbinsx=50, name='Returns',
            marker=dict(color=COLORS['blue'], opacity=0.75, line=dict(color=COLORS['bg'],width=0.3))
        ))
        x_range = np.linspace(r_vals.min()-1, r_vals.max()+1, 200)
        mu_n, sg_n = r_vals.mean(), r_vals.std()
        scale = len(r_vals)*(r_vals.max()-r_vals.min())/50
        fig_dist.add_trace(go.Scatter(
            x=x_range, y=stats.norm.pdf(x_range,mu_n,sg_n)*scale,
            mode='lines', line=dict(color=COLORS['orange'],width=2,dash='dash'), name='Normal'))
        fig_dist.add_vline(x=metrics_port['var_95']*100,
            line=dict(color=COLORS['red'],width=1.5,dash='dot'),
            annotation_text='VaR 95%', annotation_font_color=COLORS['red'])
        fig_dist.update_layout(
            title=f'Return distribution  ·  Skew={metrics_port["skew"]:.2f}  Kurt={metrics_port["kurt"]:.2f}',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=11),
            xaxis=dict(gridcolor=COLORS['border'],ticksuffix='%'),
            yaxis=dict(gridcolor=COLORS['border']),
            barmode='overlay', margin=dict(t=45,b=30,l=50,r=20)
        )

        risk_table_data = [
            {'Metric':'Sharpe ratio',    'Portfolio':f"{metrics_port['sharpe']:.3f}",  'Benchmark': f"{metrics_bench['sharpe']:.3f}" if metrics_bench else '-'},
            {'Metric':'Sortino ratio',   'Portfolio':f"{metrics_port['sortino']:.3f}", 'Benchmark': f"{metrics_bench['sortino']:.3f}" if metrics_bench else '-'},
            {'Metric':'Calmar ratio',    'Portfolio':f"{metrics_port['calmar']:.3f}",  'Benchmark': '-'},
            {'Metric':'Max Drawdown',    'Portfolio':f"{metrics_port['max_dd']:.2%}",  'Benchmark': f"{metrics_bench['max_dd']:.2%}" if metrics_bench else '-'},
            {'Metric':'VaR 95%',         'Portfolio':f"{metrics_port['var_95']:.2%}",  'Benchmark': f"{metrics_bench['var_95']:.2%}" if metrics_bench else '-'},
            {'Metric':'CVaR 95%',        'Portfolio':f"{metrics_port['cvar_95']:.2%}", 'Benchmark': f"{metrics_bench['cvar_95']:.2%}" if metrics_bench else '-'},
            {'Metric':'VaR 99%',         'Portfolio':f"{metrics_port['var_99']:.2%}",  'Benchmark': '-'},
            {'Metric':'CVaR 99%',        'Portfolio':f"{metrics_port['cvar_99']:.2%}", 'Benchmark': '-'},
            {'Metric':'Ann. Volatility', 'Portfolio':f"{metrics_port['ann_vol']:.2%}", 'Benchmark': f"{metrics_bench['ann_vol']:.2%}" if metrics_bench else '-'},
            {'Metric':'Skewness',        'Portfolio':f"{metrics_port['skew']:.3f}",    'Benchmark': '-'},
            {'Metric':'Excess Kurtosis', 'Portfolio':f"{metrics_port['kurt']:.3f}",    'Benchmark': '-'},
        ]
        risk_tbl = dash_table.DataTable(
            data=risk_table_data,
            columns=[{'name':'Metric','id':'Metric'},{'name':'Portfolio','id':'Portfolio'},{'name':'Benchmark','id':'Benchmark'}],
            style_table={'overflowX':'auto'},
            style_header={'backgroundColor':COLORS['card2'],'color':COLORS['sub'],
                          'fontSize':'11px','fontWeight':'600','letterSpacing':'1px','border':'none'},
            style_cell={'backgroundColor':COLORS['card'],'color':COLORS['text'],
                        'fontSize':'13px','border':f'1px solid {COLORS["border"]}','padding':'9px 14px'},
            style_data_conditional=[
                {'if':{'row_index':'odd'},'backgroundColor':COLORS['card2']},
            ]
        )

        return html.Div([
            html.Div([dcc.Graph(figure=fig_sr), dcc.Graph(figure=fig_dist)],
                     style={'display':'grid','gridTemplateColumns':'1fr 1fr','gap':'16px'}),
            html.Div([section_title('Risk Metrics'), risk_tbl],
                     style={'marginTop':'20px','background':COLORS['card'],
                            'borderRadius':'10px','padding':'16px'})
        ])

    # ── TAB 4: ETF MAP ────────────────────────────────────────
    elif tab == 'tab-etf':
        df_plot = top_etf_plot.copy()
        df_plot['color'] = df_plot['Categoria'].map(CAT_PAL).fillna(COLORS['sub'])
        df_plot['size']  = (df_plot['DeltaR2'].clip(lower=0)*20000 + 8).clip(upper=40)

        fig_scatter = go.Figure()
        for cat in df_plot['Categoria'].unique():
            sub = df_plot[df_plot['Categoria']==cat]
            fig_scatter.add_trace(go.Scatter(
                x=sub['R2_FF5'], y=sub['DeltaR2']*1000,
                mode='markers+text', name=cat,
                text=sub['Ticker'], textposition='top center',
                textfont=dict(size=9, color=COLORS['sub']),
                marker=dict(size=sub['size'], color=CAT_PAL.get(cat,COLORS['sub']),
                            opacity=0.8, line=dict(color=COLORS['bg'],width=1)),
                customdata=np.stack([sub['Nome'],sub['Alpha_ETF']], axis=-1),
                hovertemplate='<b>%{text}</b> — %{customdata[0]}<br>R²(FF5)=%{x:.3f}  ΔR²=%{y:.4f}‰<br>Alpha ETF=%{customdata[1]:.2f}%<extra></extra>'
            ))
        fig_scatter.add_vline(x=0.7, line=dict(color=COLORS['orange'],width=1,dash='dot'),
            annotation_text='High redundancy', annotation_font_color=COLORS['orange'])
        fig_scatter.add_hline(y=0, line=dict(color=COLORS['sub'],width=1))
        fig_scatter.update_layout(
            title='ETF Factor Map: FF5 redundancy vs incremental information content',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=10),
            xaxis=dict(title='R²(ETF vs FF5) — redundancy with academic factors',
                       gridcolor=COLORS['border'], range=[-0.02,1.02]),
            yaxis=dict(title='Incremental ΔR² (×1000)', gridcolor=COLORS['border']),
            legend=dict(bgcolor='rgba(0,0,0,0)', x=1.01, y=1),
            height=500, margin=dict(t=50,b=50,l=60,r=120)
        )

        r2c = df_etf_stats.groupby('Categoria')['DeltaR2'].sum().sort_values(ascending=True)
        cat_display = {'us_sector':'US Sector','smart_beta':'Smart Beta',
                       'eu_equity':'EU Equity','em_equity':'EM Equity',
                       'alternatives':'Alternatives','fixed_income':'Fixed Income','thematic':'Thematic'}
        r2c.index = [cat_display.get(i,i) for i in r2c.index]

        fig_r2bar = go.Figure(go.Bar(
            x=r2c.values, y=r2c.index, orientation='h',
            marker=dict(color=[CAT_PAL.get(k.lower().replace(' ','_'),COLORS['blue'])
                               for k in r2c.index], opacity=0.85),
            text=[f'+{v:.5f}' for v in r2c.values],
            textposition='outside', textfont=dict(color=COLORS['sub'],size=10)
        ))
        fig_r2bar.update_layout(
            title='Incremental ΔR² by ETF category',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=10),
            xaxis=dict(gridcolor=COLORS['border']),
            yaxis=dict(gridcolor='rgba(0,0,0,0)'),
            margin=dict(t=45,b=30,l=120,r=80), height=280
        )

        return html.Div([
            dcc.Graph(figure=fig_scatter),
            dcc.Graph(figure=fig_r2bar, style={'marginTop':'16px'})
        ])

    # ── TAB 5: HEATMAP ────────────────────────────────────────
    elif tab == 'tab-heat':
        z_vals = pivot_heat.values
        fig_hm = go.Figure(go.Heatmap(
            z=z_vals,
            x=month_names[:z_vals.shape[1]],
            y=[str(y) for y in pivot_heat.index],
            colorscale=[[0,'#b91c1c'],[0.5,'#1f2937'],[1,'#065f46']],
            zmid=0, zmin=-8, zmax=8,
            text=[[f'{v:+.2f}%' for v in row] for row in z_vals],
            texttemplate='%{text}', textfont=dict(size=12),
            hovertemplate='%{y} %{x}: %{z:+.2f}%<extra></extra>',
            colorbar=dict(title='Ret. %', ticksuffix='%',
                          tickfont=dict(color=COLORS['text']),
                          titlefont=dict(color=COLORS['text']))
        ))
        fig_hm.update_layout(
            title='Monthly portfolio returns (%)',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card'],
            font=dict(color=COLORS['text'],size=12),
            xaxis=dict(side='top'), margin=dict(t=80,b=30,l=60,r=60),
            height=max(300, len(pivot_heat)*55+120)
        )

        annual_ret = returns['PORTFOLIO'].resample('YE').apply(lambda x:(1+x).prod()-1)*100
        fig_ann = go.Figure(go.Bar(
            x=[str(d.year) for d in annual_ret.index],
            y=annual_ret.values,
            marker=dict(color=[COLORS['green'] if v>=0 else COLORS['red'] for v in annual_ret.values],
                        opacity=0.85),
            text=[f'{v:+.1f}%' for v in annual_ret.values],
            textposition='outside', textfont=dict(color=COLORS['text'],size=11)
        ))
        fig_ann.add_hline(y=0, line=dict(color=COLORS['sub'],width=1))
        fig_ann.update_layout(
            title='Annual return (%)',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=11),
            yaxis=dict(gridcolor=COLORS['border'],ticksuffix='%'),
            xaxis=dict(gridcolor='rgba(0,0,0,0)'),
            margin=dict(t=45,b=30,l=60,r=20), height=280
        )

        return html.Div([
            dcc.Graph(figure=fig_hm),
            dcc.Graph(figure=fig_ann, style={'marginTop':'16px'})
        ])

    # ── TAB 6: REGIONS ────────────────────────────────────────
    elif tab == 'tab-regions':
        fig_reg = go.Figure()
        fig_reg.add_trace(go.Scatter(
            x=cum_port.index, y=cum_port.values,
            name='Total Portfolio',
            line=dict(color=COLORS['text'],width=3), opacity=0.9
        ))
        for region, rseries in regional_returns.items():
            cum_r = (1 + pd.Series(rseries, index=returns.index).fillna(0)).cumprod()
            fig_reg.add_trace(go.Scatter(
                x=cum_r.index, y=cum_r.values,
                name=f'{region} ({region_weights.get(region,0):.0%})',
                line=dict(color=reg_colors.get(region,COLORS['sub']),
                          width=1.8, dash='dash')
            ))
        fig_reg.update_layout(
            title='Cumulative performance by geographic region',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=11),
            legend=dict(bgcolor='rgba(0,0,0,0)', orientation='h', y=1.1),
            xaxis=dict(gridcolor=COLORS['border']),
            yaxis=dict(gridcolor=COLORS['border']),
            hovermode='x unified', margin=dict(t=60,b=30,l=50,r=20)
        )

        fig_pie = go.Figure(go.Pie(
            labels=list(region_weights.keys()),
            values=list(region_weights.values()),
            hole=0.5,
            marker=dict(colors=[reg_colors.get(r,COLORS['sub']) for r in region_weights],
                        line=dict(color=COLORS['bg'],width=2)),
            textfont=dict(size=11)
        ))
        fig_pie.add_annotation(text='Regions', x=0.5, y=0.5, showarrow=False,
                                font=dict(size=13, color=COLORS['text']))
        fig_pie.update_layout(
            title='Regional portfolio composition',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card'],
            font=dict(color=COLORS['text'],size=11),
            margin=dict(t=45,b=20,l=20,r=20), height=320
        )

        reg_sharpes = {}
        for region, rseries in regional_returns.items():
            rs = pd.Series(rseries)
            reg_sharpes[region] = round(rs.mean()/rs.std()*freq_mult**0.5, 3)
        fig_sharpe_reg = go.Figure(go.Bar(
            x=list(reg_sharpes.keys()), y=list(reg_sharpes.values()),
            marker=dict(color=[reg_colors.get(r,COLORS['sub']) for r in reg_sharpes], opacity=0.85),
            text=[f'{v:.3f}' for v in reg_sharpes.values()],
            textposition='outside', textfont=dict(color=COLORS['text'])
        ))
        fig_sharpe_reg.add_hline(y=0, line=dict(color=COLORS['sub'],width=1))
        fig_sharpe_reg.update_layout(
            title='Sharpe ratio by region',
            paper_bgcolor=COLORS['card'], plot_bgcolor=COLORS['card2'],
            font=dict(color=COLORS['text'],size=11),
            yaxis=dict(gridcolor=COLORS['border']),
            xaxis=dict(gridcolor='rgba(0,0,0,0)'),
            margin=dict(t=45,b=30,l=50,r=20), height=280
        )

        return html.Div([
            dcc.Graph(figure=fig_reg),
            html.Div([dcc.Graph(figure=fig_pie), dcc.Graph(figure=fig_sharpe_reg)],
                     style={'display':'grid','gridTemplateColumns':'1fr 1fr',
                            'gap':'16px','marginTop':'16px'})
        ])

    return html.Div('Tab not found')

print(f'\n✅ Dashboard configured — 6 sections, {len(returns)} periods loaded.')
print(f'   Run the next cell to launch.')



✅ Dashboard configured — 6 sections, 330 periods loaded.
   Run the next cell to launch.


In [95]:
# ============================================================
# CELL 11 — Dashboard launch
# ============================================================

print(f'🚀 Launching dashboard at http://localhost:{DASHBOARD_PORT}')
print('   Press Ctrl+C (or interrupt the kernel) to stop.\n')

# In Jupyter use jupyter_mode to avoid blocking
try:
    app.run(debug=False, port=DASHBOARD_PORT, jupyter_mode='tab')
except Exception:
    # Fallback for other environments
    app.run(debug=False, port=DASHBOARD_PORT)


🚀 Launching dashboard at http://localhost:8050
   Press Ctrl+C (or interrupt the kernel) to stop.

Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>